# Ядро редукций Max-Cut → Ising

**Версия 0.1.0a2.** Этот ноутбук проверяет точное парное ограничение `s_u = σ s_v`: рабочий граф, накопленный сдвиг, полную константу Изинга, обратное восстановление и снимок состояния.

Используется энергетическая конвенция `E = -C` для Max-Cut и `E = c + Σ h_i s_i + Σ J_ij s_i s_j` для внутренней задачи. Ноутбук не запускает QAOA, не выбирает пару и не измеряет корреляции.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
from itertools import product
import json
import math

from QA_max_cut_lib import GraphReducer, IsingProblem, WeightedGraph, __version__

root = Path.cwd()
if not (root / 'configs').is_dir():
    root = root.parent
assert (root / 'configs').is_dir(), 'Open the notebook from the project or notebooks folder.'
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:6]
output_dir = root / 'notebooks' / '_outputs' / ('reduction_' + stamp)
output_dir.mkdir(parents=True, exist_ok=True)
print('QA_max_cut_lib', __version__)
print('Output:', output_dir)

def cut(graph, spins):
    return math.fsum(w for u, v, w in graph.edges if spins[u] != spins[v])

def energy(problem, spins):
    terms = [problem.constant]
    terms.extend(problem.field(node) * spins[node] for node in problem.nodes)
    terms.extend(J * spins[u] * spins[v] for (u, v), J in problem.couplings.items())
    return math.fsum(terms)

def assignments(nodes):
    for spins in product((-1, 1), repeat=len(nodes)):
        yield dict(zip(nodes, spins))


QA_max_cut_lib 0.1.0a2
Output: C:\Users\Alexey\PycharmProjects\QA_max_cut_lib\notebooks\_outputs\reduction_20261102T124023Z_90116b


## 1. Контрольный граф
Вершины `[10, 30, 80, 101]`, вершина `101` изолирована. Удаляем `10`, сохраняем `30`, σ = −1.

In [2]:
graph = WeightedGraph(
    edges=[(10, 30, 2.0), (10, 80, 3.0), (30, 80, 5.0)],
    nodes=[10, 30, 80, 101],
)
reducer = GraphReducer()
state = reducer.start(graph)
print('C initial:', cut(graph, {node: 1 for node in graph.nodes}))
print('Ising constant:', state.ising.constant)
print('Ising couplings:', state.ising.couplings)
print('Cut offset:', state.cut_offset)
print('Working nodes:', state.current_nodes)


C initial: 0.0
Ising constant: -5.0
Ising couplings: {(10, 30): 1.0, (10, 80): 1.5, (30, 80): 2.5}
Cut offset: 0.0
Working nodes: (10, 30, 80, 101)


## 2. Перевод графа в Изинг
Проверяем `E_Ising = −C` для всех 2⁴ назначений исходного графа.

In [3]:
initial_problem = IsingProblem.from_maxcut(graph)
errors = [abs(energy(initial_problem, spins) + cut(graph, spins))
          for spins in assignments(graph.nodes)]
print('max |E_Ising + C|:', max(errors))
assert max(errors) < 1e-12


max |E_Ising + C|: 0.0


## 3. Одно слияние
Событие содержит вес выбранного ребра, фиксированную энергию, сдвиг разреза и полную константу Изинга. Это разные величины.

In [4]:
event = reducer.reduce(state, 10, 30, -1, correlation=-0.6)
for key in ('selected_weight', 'fixed_energy', 'expected_fixed_energy',
            'constraint_probability', 'cut_offset_delta', 'cut_offset',
            'ising_constant_before', 'ising_constant', 'ising_constant_delta',
            'edges_after', 'mean_abs_weight'):
    print(f'{key}: {event[key]}')
working = state.current_graph()
print('Working graph:', working.nodes, working.edges)
assert event['fixed_energy'] == -2.0
assert event['cut_offset'] == 5.0
assert event['ising_constant'] == -6.0
assert math.isclose(event['expected_fixed_energy'], -1.6)


selected_weight: 2.0
fixed_energy: -2.0
expected_fixed_energy: -1.6
constraint_probability: 0.8
cut_offset_delta: 5.0
cut_offset: 5.0
ising_constant_before: -5.0
ising_constant: -6.0
ising_constant_delta: -1.0
edges_after: 1
mean_abs_weight: 2.0
Working graph: (30, 80, 101) ((30, 80, 2.0),)


## 4. Равенство энергий на всём остатке
Для каждого назначения `[30, 80, 101]` подставляем `s_10 = −s_30` и сравниваем исходную и редуцированную энергии.

In [5]:
differences = []
for residual in assignments(state.current_nodes):
    before = {10: -residual[30], **residual}
    differences.append(abs(energy(initial_problem, before) -
                           energy(state.ising, residual)))
print('checked assignments:', len(differences))
print('max energy difference:', max(differences))
assert max(differences) < 1e-12


checked assignments: 8
max energy difference: 0.0


## 5. Восстановление решения
Остаток `{30: +1, 80: −1, 101: +1}` восстанавливается в исходные номера.

In [6]:
residual = {30: 1, 80: -1, 101: 1}
recovered = reducer.restore(state, residual)
print('Recovered:', recovered)
print('C original:', state.original_cut_value(recovered))
print('E original:', state.original_energy(recovered))
assert state.original_cut_value(recovered) == 7.0
assert state.original_energy(recovered) == -7.0


Recovered: {30: 1, 80: -1, 101: 1, 10: -1}
C original: 7.0
E original: -7.0


## 6. Цепочка редукций
Второй шаг удаляет `80`, сохраняя `30`. Вершина `30` уже была сохранена на первом шаге; восстановление идёт в обратном порядке.

In [7]:
chain = GraphReducer()
chain_state = chain.start(graph)
chain.reduce(chain_state, 10, 30, -1)
chain.reduce(chain_state, 80, 30, 1)
print('constraints:', chain_state.constraints)
print('working nodes:', chain_state.current_nodes)
print('working graph:', chain_state.current_graph().edges)
print('cut offset:', chain_state.cut_offset)
print('Ising constant:', chain_state.ising.constant)
chain_residual = {30: 1, 101: -1}
chain_recovered = chain.restore(chain_state, chain_residual)
print('recovered:', chain_recovered)
print('C original:', chain_state.original_cut_value(chain_recovered))
assert chain_state.original_cut_value(chain_recovered) == 5.0


constraints: (SpinConstraint(removed=10, retained=30, sign=-1), SpinConstraint(removed=80, retained=30, sign=1))
working nodes: (30, 101)
working graph: ()
cut offset: 5.0
Ising constant: -5.0
recovered: {30: 1, 101: -1, 80: 1, 10: -1}
C original: 5.0


## 7. Снимок состояния
Сохраняем математическое состояние в JSON, восстанавливаем его и продолжаем редукцию. Это не checkpoint оптимизатора.

In [8]:
snapshot = state.snapshot()
snapshot_path = output_dir / 'reduction_state.json'
snapshot_path.write_text(json.dumps(snapshot, ensure_ascii=False,
                                     allow_nan=False, indent=2), encoding='utf-8')
loaded = json.loads(snapshot_path.read_text(encoding='utf-8'))
restored_state = reducer.from_snapshot(loaded)
print('snapshot version:', snapshot['version'])
print('energy convention:', snapshot['energy_convention'])
print('constraints:', restored_state.constraints)
continued_event = reducer.reduce(restored_state, 80, 30, 1)
print('continued cut offset:', continued_event['cut_offset'])
print('continued Ising constant:', continued_event['ising_constant'])
print('saved:', snapshot_path)
assert restored_state.restore({30: 1, 101: -1}) == chain_recovered


snapshot version: 1
energy convention: E(s) = c + sum_i h_i s_i + sum_{i<j} J_ij s_i s_j; for Max-Cut E = -C and C = sum_e w_e (1 - s_u s_v) / 2
constraints: (SpinConstraint(removed=10, retained=30, sign=-1),)
continued cut offset: 5.0
continued Ising constant: -5.0
saved: C:\Users\Alexey\PycharmProjects\QA_max_cut_lib\notebooks\_outputs\reduction_20261102T124023Z_90116b\reduction_state.json


## Итог
Проверены точное равенство энергий, связь `C_original = b + C_reduced`, различие `fixed_energy`, `Δb` и `Δc`, обратное восстановление и JSON-round-trip математического состояния. Реальные корреляции и выбор пары появятся в следующих этапах.